# Physical inputs, coordinates and saved results

Models take inputs in **physical** units. The active subspace is defined in
**normalized** coordinates, where every input has the same scale. This tutorial uses the
borehole function to show

- how an input space maps between the two,
- what happens to gradients and directions,
- how the input density changes the result,
- how to save a result, reload it, and read its provenance.

See {doc}`../user_guide/input_spaces` and {doc}`../user_guide/saving`.

## The borehole function

The borehole function computes the water flow rate through a borehole from eight
inputs. In the built-in version every input is uniform between physical bounds. This
is a variant: the borehole problem is often stated with a normal $r_w$ and a log-normal
$r$ (as in the 2016 tutorial), and an input space here is either all uniform or all
Gaussian.

In [ ]:
import json
import tempfile
from pathlib import Path

import numpy as np

import active_subspaces as ac

tf = ac.test_functions
borehole = tf.BOREHOLE
space = borehole.space
for name, unit, lo, hi, text in zip(
    space.names,
    space.units,
    space.lower,
    space.upper,
    borehole.input_descriptions,
    strict=True,
):
    print(f"{name:>3} [{unit:>6}]  {lo:>8g} .. {hi:<8g} {text}")

## Coordinates

For a uniform space with bounds $\ell \le \xi \le u$ (physical), the normalized
coordinates are

$$x = 2\,\frac{\xi - \ell}{u - \ell} - 1 \in [-1, 1]^m .$$

`to_normalized` and `from_normalized` apply the map and its inverse. You always pass
physical arrays to the library; the normalized ones appear inside results such as
`sub.W1`.

In [ ]:
X = space.sample(400, rng=1)  # physical
x = space.to_normalized(X)
print("physical range of r: ", X[:, 1].min().round(1), "..", X[:, 1].max().round(1))
print("normalized range:    ", x.min().round(3), "..", x.max().round(3))
print("round trip exact:    ", np.allclose(space.from_normalized(x), X))

A point outside the bounds is an error, not an extrapolation. Each library error has a
stable `code` and structured `details`:

In [ ]:
bad = space.lower.copy()
bad[0] = 0.2  # rw above its upper bound 0.15 m
try:
    ac.evaluate(borehole, X=bad[None, :])
except ac.InputError as err:
    print(err.code, dict(err.details))

## Gradients

By the chain rule, the gradient with respect to $x$ is the physical gradient times the
half-ranges: $\partial f/\partial x_j = \tfrac12 (u_j - \ell_j)\, \partial f/\partial \xi_j$.
Physical derivatives depend on units; normalized ones do not. Compare `rw` and `Kw`
below: their median physical derivatives differ by a factor of about $2\times10^5$,
because `rw` varies over 0.1 m and `Kw` over about 2200 m/yr. Their normalized
derivatives differ by a factor of about 10, which is the comparison that matters.

In [ ]:
f, G = ac.evaluate_with_grads(borehole, X=X)  # G is physical
Gn = space.grads_to_normalized(G)
print(f"{'input':>5}{'median |df/dxi|':>18}{'median |df/dx|':>17}")
for name, gp, gn in zip(space.names, np.median(abs(G), 0), np.median(abs(Gn), 0), strict=True):
    print(f"{name:>5}{gp:>18.3e}{gn:>17.3e}")

`ac.active_subspace` does this conversion itself when you pass physical gradients with
the space. Do not pass `Gn` with the same space, or the gradients are scaled twice.

In [ ]:
sub = ac.active_subspace(grads=G, space=space, nboot=100, rng=2)
print("n =", sub.n)


def show(values, fmt):
    return "  ".join(f"{k}={v:{fmt}}" for k, v in zip(space.names, values, strict=True))


print("w_1 (normalized):", show(sub.W1[:, 0], ".3f"))

## Directions in physical units

$w_1$ is a direction in normalized coordinates. `sub.physical_directions()` returns the
corresponding unit direction in physical coordinates: since
$w_1^\top x = w_1^\top D^{-1}(\xi - c)$ with $D = \operatorname{diag}((u - \ell)/2)$ and
$c$ the centre, it is $D^{-1} w_1$, normalized. It answers "which way do I move the
physical inputs to change $f$ fastest per unit of input". Because it mixes units, an
input with a small range (here `rw`, a tenth of a metre wide) dominates it. Use the
normalized $w_1$ to rank the inputs by importance.

In [ ]:
print("physical direction:", show(sub.physical_directions()[:, 0], ".1e"))
Y, Z = ac.forward_map(sub, X=X)  # active and inactive variables, from physical X
print("y = W1^T x:", Y.shape, " z = W2^T x:", Z.shape)

## The input density matters

The active subspace belongs to the pair ($f$, input density), not to $f$ alone. To see
this, describe the same eight inputs as independent Gaussians centred in the box with
standard deviation one eighth of the range, and wrap the same Python functions
`tf.borehole` and `tf.borehole_grad` in a new `ac.Model`. The box bounds lie four
standard deviations from the centre. Gaussian draws can leave the box; the borehole
formula stays finite there as long as $r > r_w$, which holds for these draws.

For a Gaussian space with mean $\mu$ and covariance $\Sigma = LL^\top$, the normalized
coordinates are $x = L^{-1}(\xi - \mu)$. Here both spaces are centred at the same point,
and their scales differ by the same factor 4 in every input, so normalized directions
can be compared directly.

In [ ]:
centre = (space.lower + space.upper) / 2
sd = (space.upper - space.lower) / 8
gauss = ac.Gaussian(mean=centre, cov=sd**2, names=space.names, units=space.units)
borehole_g = ac.Model(
    space=gauss,
    value=tf.borehole,  # batch function: (k, 8) physical -> (k,)
    grad=tf.borehole_grad,  # (k, 8) physical -> (k, 8)
    name="borehole_gaussian",
    qoi_units="m^3/yr",
)
Xg = gauss.sample(400, rng=3)
sub_g = ac.active_subspace(grads=ac.evaluate_grads(borehole_g, X=Xg), space=gauss, rng=4)

print("uniform  eigenvalues:", np.array2string(sub.eigenvalues[:3], precision=3))
print("gaussian eigenvalues:", np.array2string(sub_g.eigenvalues[:3], precision=3))
U, V = sub.W1, sub_g.W1
print(f"distance between the two w_1: {np.linalg.norm(U @ U.T - V @ V.T, 2):.3f}")

The eigenvalues shrink by roughly the square of the scale factor 4, since normalized
gradients scale with the input spread, and the direction moves a little.

**Mixed densities.** An input space is either all uniform (`ac.Uniform`) or all
Gaussian (`ac.Gaussian`). For inputs with mixed or other densities, transform them to
one of these by hand and apply the chain rule to the gradients yourself; the
{doc}`migration guide <../migration>` (example E4) works through the 2016 borehole, which
had a Gaussian and a log-normal input.

## Saving, loading and provenance

Every result can be saved to a NumPy `.npz` file (no pickle) and loaded back with
`ac.load`. The loaded result compares equal to the original. Here the file lives in a
temporary directory.

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "borehole_subspace.npz"
    sub.save(path)
    back = ac.load(path)
    print(type(back).__name__, "| equal to the original:", back == sub)

Each result records how it was made in `provenance`: the function, its parameters (input
arrays are recorded by shape, dtype and SHA-256 digest, not stored), the input space,
the random-number state on entry, and the library and platform versions.

In [ ]:
prov = back.provenance
print("function:  ", prov.function)
print("parameters:", json.dumps(prov.to_dict()["parameters"], indent=None)[:200], "...")
print("rng:       ", prov.rng)
print(
    f"versions:   library {prov.library_version}, numpy {prov.numpy_version}, "
    f"scipy {prov.scipy_version}"
)

For JSON, `to_dict` gives a plain dictionary and `ac.from_dict` rebuilds the result:

In [ ]:
text = json.dumps(sub.to_dict(), allow_nan=False)
again = ac.from_dict(json.loads(text))
print(f"{len(text)} characters of JSON; round trip equal: {again == sub}")

{doc}`reproducibility` shows how to use the recorded random state to rerun a
computation.

## References

- Surjanovic, S., and Bingham, D. Virtual Library of Simulation Experiments: Test
  Functions and Datasets. <https://www.sfu.ca/~ssurjano/>.